# PROYECTO FINAL

In [5]:
# =========================================
# 1. LIBRERÍAS
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.style.use('ggplot')


# =========================================
# 2. CARGA DE DATOS
# =========================================
def cargar_datos():
    try:
        archivos = [
            "clientes.csv",
            "productos.csv",
            "ventas.csv",
            "inventario.csv",
            "entregas.csv",
            "devoluciones.csv"
        ]

        for archivo in archivos:
            if not os.path.exists(archivo):
                print(f"❌ Falta el archivo: {archivo}")
                return None, None, None, None, None, None

        clientes = pd.read_csv("clientes.csv")
        productos = pd.read_csv("productos.csv")
        ventas = pd.read_csv("ventas.csv")
        inventario = pd.read_csv("inventario.csv")
        entregas = pd.read_csv("entregas.csv")
        devoluciones = pd.read_csv("devoluciones.csv")

        print("✅ Datos cargados correctamente")
        return clientes, productos, ventas, inventario, entregas, devoluciones

    except Exception as e:
        print("❌ Error al cargar datos:", e)
        return None, None, None, None, None, None


clientes, productos, ventas, inventario, entregas, devoluciones = cargar_datos()


# =========================================
# 3. BITÁCORA
# =========================================
if clientes is not None:
    print("Clientes:", clientes.shape)
    print(clientes.dtypes)
    print(clientes.head())


# =========================================
# 4. VALIDACIÓN
# =========================================
if ventas is not None:
    print("\n🔍 Nulos en ventas:\n", ventas.isnull().sum())

if clientes is not None:
    print("\n🔁 Duplicados clientes:", clientes.duplicated(subset="cliente_id").sum())

if productos is not None:
    print("\n💸 Precios inválidos:\n", productos[productos["precio"] <= 0])

if ventas is not None:
    print("\n📦 Cantidades inválidas:\n", ventas[ventas["cantidad"] <= 0])

if ventas is not None:
    print("\n📅 Fechas inválidas:",
          pd.to_datetime(ventas["fecha"], errors="coerce").isnull().sum())

if ventas is not None and productos is not None:
    print("\n🔗 Ventas sin producto:",
          ventas[~ventas["producto_id"].isin(productos["producto_id"])])

if ventas is not None and clientes is not None:
    print("\n🔗 Ventas sin cliente:",
          ventas[~ventas["cliente_id"].isin(clientes["cliente_id"])])

if devoluciones is not None and ventas is not None:
    print("\n🔗 Devoluciones sin venta:",
          devoluciones[~devoluciones["venta_id"].isin(ventas["venta_id"])])


# =========================================
# 5. LIMPIEZA
# =========================================
def limpiar_columnas(df):
    df.columns = df.columns.str.lower()
    return df

def convertir_fecha(df, col):
    df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

if clientes is not None:
    clientes = limpiar_columnas(clientes)
    clientes = clientes.drop_duplicates(subset="cliente_id")

if productos is not None:
    productos = limpiar_columnas(productos)
    productos = productos[productos["precio"] > 0]

if ventas is not None:
    ventas = limpiar_columnas(ventas)
    ventas = ventas[ventas["cantidad"] > 0]
    ventas = convertir_fecha(ventas, "fecha")


# =========================================
# 6. INTEGRACIÓN
# =========================================
def integrar(clientes, productos, ventas, entregas, devoluciones):
    df = ventas.merge(clientes, on="cliente_id", how="left")
    df = df.merge(productos, on="producto_id", how="left")
    df = df.merge(entregas, on="venta_id", how="left")
    df = df.merge(devoluciones, on="venta_id", how="left")
    return df

if clientes is not None:
    df = integrar(clientes, productos, ventas, entregas, devoluciones)


# =========================================
# 7. TRANSFORMACIÓN
# =========================================
def crear_variables(df):
    df["ingreso_bruto"] = df["cantidad"] * df["precio"]
    df["descuento_aplicado"] = df["ingreso_bruto"] * df["descuento"]
    df["ingreso_neto"] = df["ingreso_bruto"] - df["descuento_aplicado"]
    df["costo_total"] = df["cantidad"] * df["costo"]
    df["ganancia"] = df["ingreso_neto"] - df["costo_total"]
    df["margen"] = df["ganancia"] / df["ingreso_neto"]

    df["mes"] = df["fecha"].dt.month
    df["trimestre"] = df["fecha"].dt.quarter

    return df

df = crear_variables(df)


# =========================================
# 8. CLASIFICACIÓN
# =========================================
def clasificar_cliente(edad):
    if edad < 30:
        return "joven"
    elif edad < 60:
        return "adulto"
    else:
        return "mayor"

df["tipo_cliente"] = df["edad"].apply(clasificar_cliente)


# =========================================
# 9. ALERTAS
# =========================================
def generar_alertas(df, inventario):
    alertas = {}

    alertas["bajo_stock"] = inventario[inventario["stock_actual"] < inventario["stock_minimo"]]
    alertas["perdidas"] = df[df["ganancia"] < 0]
    alertas["entregas_lentas"] = df[df["tiempo_entrega_dias"] > 5]

    return alertas

alertas = generar_alertas(df, inventario)

print("\n🚨 Alertas stock:\n", alertas["bajo_stock"].head())


# =========================================
# 10. ANÁLISIS
# =========================================
print("\n💰 Ingreso total:", df["ingreso_neto"].sum())

print("\n🏆 Top productos:")
print(df.groupby("nombre_producto")["ingreso_neto"].sum().sort_values(ascending=False).head())

print("\n👑 Top clientes:")
print(df.groupby("nombre")["ingreso_neto"].sum().sort_values(ascending=False).head())


# =========================================
# 11. VISUALIZACIONES
# =========================================
df.groupby("nombre_producto")["ingreso_neto"].sum().plot(kind="bar", title="Ingresos por producto")
plt.show()

df.groupby("fecha")["ingreso_neto"].sum().plot(title="Ventas en el tiempo")
plt.show()

df.groupby("categoria")["ingreso_neto"].sum().plot(kind="pie", autopct="%1.1f%%")
plt.title("Categorías")
plt.show()

df["margen"].hist()
plt.title("Distribución margen")
plt.show()

df.groupby("mes")["ingreso_neto"].sum().plot(title="Ventas por mes")
plt.show()

df.groupby("tipo_cliente")["ingreso_neto"].sum().plot(kind="bar", title="Clientes")
plt.show()


# =========================================
# 12. EXPORTACIÓN
# =========================================
df.to_csv("dataset_maestro.csv", index=False)

df.groupby("nombre_producto")["ingreso_neto"].sum().to_csv("reporte_productos.csv")


# =========================================
# 13. CONCLUSIONES
# =========================================
print("""
CONCLUSIONES:

- Se detectaron problemas de calidad en los datos
- Hay productos con baja rentabilidad
- Existe riesgo de bajo inventario
- Algunas entregas son tardías

RECOMENDACIONES:
- Mejorar control de datos
- Ajustar precios
- Optimizar inventario
- Mejorar logística
""")

Saving clientes.csv to clientes.csv
Error al cargar datos: [Errno 2] No such file or directory: 'productos.csv'
Clientes data is not loaded.
Ventas data is not loaded, cannot check for nulls.
Clientes data is not loaded, cannot check for duplicates.
Productos data is not loaded, cannot check for invalid prices.
Ventas data is not loaded, cannot check for negative quantities.
Ventas data is not loaded, cannot check for invalid dates.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')